In [1]:
import pandas as pd
import requests
import re
import json
import time
import io
from nltk.tokenize import word_tokenize
import nltk

# Configuration
USER_AGENT = "DTU_Social_Science_Project/ModularScraper (sXXXXXX@student.dtu.dk)"
# Adjust this range if you need to re-run specific years
year_list = range(2001, 2026) 

def fetch_wikipedia_text(session, title, iata, year):
    url = "https://en.wikipedia.org/w/api.php"
    for t in [title, f"{iata} airport"]:
        params = {
            "action": "query", "prop": "revisions", "titles": t,
            "redirects": 1, "rvlimit": 1, "format": "json",
            "rvprop": "content", "rvdir": "older",
            "rvstart": f"{year}-12-31T23:59:59Z"
        }
        try:
            r = session.get(url, params=params, timeout=15)
            data = r.json()
            pages = data.get("query", {}).get("pages", {})
            for pid in pages:
                if pid != "-1" and "revisions" in pages[pid]:
                    return pages[pid]["revisions"][0]["*"]
        except: continue
    return ""

# Get Airport List
headers = {"User-Agent": USER_AGENT}
r_main = requests.get("https://en.wikipedia.org/wiki/List_of_airports_in_the_United_States", headers=headers)
df = pd.read_html(io.StringIO(r_main.text), match="IATA")[0].dropna(subset=['IATA'])
airports = df.to_dict('records')

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

for year in year_list:
    year_data = [] # THIS IS THE KEY: We empty the RAM for every new year
    print(f"\n--- SCRAPING YEAR: {year} ---")
    
    for count, airport in enumerate(airports):
        name = re.sub(r'\(.*?\)|\[.*?\]', '', str(airport['Airport'])).strip()
        iata = airport['IATA']
        print(f"[{year}] {count+1}/{len(airports)}: {name} ({iata})       ", end="\r")

        try:
            raw_text = fetch_wikipedia_text(session, name, iata, year)
            # 100% Raw Tokenization as per your request[cite: 2]
            tokens = word_tokenize(raw_text) if raw_text else []
            
            year_data.append({
                "Year": year, "IATA": iata, "Airport": name, "Tokens": tokens 
            })
        except: continue
        
        time.sleep(0.1)

    # Save individual year file
    output_file = f"AIRPORT_RAW_{year}.json"
    with open(output_file, "w", encoding='utf-8') as f:
        json.dump(year_data, f, indent=4)
    print(f"\nSUCCESS: {year} saved to {output_file}. Memory cleared.")


--- SCRAPING YEAR: 2001 ---
[2001] 387/387: Henry E. Rohlsen Airport (STX)       )                      PS)        
SUCCESS: 2001 saved to AIRPORT_RAW_2001.json. Memory cleared.

--- SCRAPING YEAR: 2002 ---
[2002] 387/387: Henry E. Rohlsen Airport (STX)       )                      PS)        
SUCCESS: 2002 saved to AIRPORT_RAW_2002.json. Memory cleared.

--- SCRAPING YEAR: 2003 ---
[2003] 387/387: Henry E. Rohlsen Airport (STX)       )                      PS)        
SUCCESS: 2003 saved to AIRPORT_RAW_2003.json. Memory cleared.

--- SCRAPING YEAR: 2004 ---
[2004] 387/387: Henry E. Rohlsen Airport (STX)       )                      PS)        
SUCCESS: 2004 saved to AIRPORT_RAW_2004.json. Memory cleared.

--- SCRAPING YEAR: 2005 ---
[2005] 387/387: Henry E. Rohlsen Airport (STX)       )                      PS)        
SUCCESS: 2005 saved to AIRPORT_RAW_2005.json. Memory cleared.

--- SCRAPING YEAR: 2006 ---
[2006] 387/387: Henry E. Rohlsen Airport (STX)       )                      